# 🔍 RAG Systems (v2): Retrieval-Augmented Generation with a Chat LLM

### A hands-on tutorial for Google Colab

This is a **rebuilt, robust version** of the RAG notebook. The original relied on
`pipeline(...)[0]["generated_text"]`, which breaks across `transformers` versions
(the output key changed, causing `KeyError: 'generated_text'`).

This version fixes that in two ways:

1. It uses a different generator model — **`Qwen/Qwen2.5-0.5B-Instruct`**, a small
   modern instruction-tuned chat LLM — instead of FLAN-T5.
2. It calls the model with `tokenizer` + `model.generate()` **directly** instead of the
   high-level `pipeline`, so there is **no dependency on any output dictionary key**.
   We decode the tokens ourselves, so it works on any recent `transformers` release.

A tiny FLAN-T5 (`text2text`) fallback path is also included at the end in case you
want a seq2seq model or you're on CPU with no internet for the Qwen weights.

---

## What is RAG?

A plain LLM answers only from its frozen training weights, which causes **hallucination**
(confident wrong answers) and **staleness** (no knowledge of new or private data).
RAG fixes this by (1) **retrieving** relevant passages from a knowledge base via semantic
vector search, (2) **augmenting** the prompt with those passages, and (3) asking the LLM to
**generate** an answer grounded in them — no fine-tuning required.

```
 User Query ─▶ Embed Query ─▶ Vector Search (FAISS) ─▶ top-k chunks
                                                          │
                                                          ▼
                                   Augmented Prompt (query + context)
                                                          │
                                                          ▼
                                        Chat LLM (Qwen) ─▶ Final Answer
```


## 1. Environment setup

| Library | Role |
|---|---|
| `sentence-transformers` | Text → dense embeddings (the **R**) |
| `faiss-cpu` | Fast nearest-neighbor vector search |
| `transformers` | Loads the chat LLM that generates the answer (the **G**) |

> 💡 In Colab: `Runtime → Change runtime type → T4 GPU` for faster generation (works on CPU too).


In [1]:
# Install (quiet). No version pin needed — we avoid the fragile pipeline output key.
!pip install -q -U transformers sentence-transformers faiss-cpu accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 49.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import textwrap
import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"torch: {torch.__version__}")


Using device: cuda
torch: 2.11.0+cu128


## 2. Building a knowledge base

A RAG system is only as good as its documents. In production this would be your wiki,
PDFs, or support tickets. Here we use a small self-contained set of short articles about
the concepts this notebook teaches, so you can read the retrieved context and judge it
yourself.


In [3]:
KNOWLEDGE_BASE = [
    {"id": "doc_1", "title": "Transformer Architecture",
     "text": ("The Transformer is a neural network architecture introduced in 2017 that relies "
              "entirely on attention mechanisms instead of recurrence or convolution to process "
              "sequences. Because every token can directly attend to every other token, Transformers "
              "train in parallel across the whole sequence, enabling much larger language models than "
              "earlier recurrent architectures like LSTMs. Most modern LLMs, including BERT and GPT, "
              "are built on the Transformer block.")},
    {"id": "doc_2", "title": "The Attention Mechanism",
     "text": ("Attention lets a model decide, for each token, how much to focus on every other token "
              "when building its representation. It uses three learned projections: a Query, a Key, and "
              "a Value. The similarity between a token's Query and every other token's Key produces "
              "attention weights, used to compute a weighted sum over the Values. This captures "
              "long-range dependencies, such as a pronoun referring to a noun many words earlier.")},
    {"id": "doc_3", "title": "BERT: Bidirectional Encoder Representations",
     "text": ("BERT is an encoder-only Transformer trained with masked language modeling: random tokens "
              "are hidden and the model predicts them using context from both directions. This "
              "bidirectional context makes BERT strong at understanding tasks such as classification, "
              "named entity recognition, and question answering, but it is not designed to generate "
              "long free-form text.")},
    {"id": "doc_4", "title": "GPT: Generative Pretrained Transformer",
     "text": ("GPT is a decoder-only Transformer trained autoregressively: given all previous tokens it "
              "predicts the next one. This makes GPT-style models naturally suited to open-ended "
              "generation — chat, summarization, code completion. Unlike BERT, GPT only attends to "
              "tokens before the current position, since it generates left to right.")},
    {"id": "doc_5", "title": "Word and Sentence Embeddings",
     "text": ("An embedding is a dense vector representing the meaning of text in a high-dimensional "
              "space. Texts with similar meaning end up close together, measurable with cosine "
              "similarity. Sentence-embedding models, such as those from sentence-transformers, are "
              "trained so whole sentences map to vectors capturing semantic similarity, ideal for "
              "semantic search.")},
    {"id": "doc_6", "title": "Retrieval-Augmented Generation (RAG)",
     "text": ("RAG combines a retriever, which searches an external knowledge base for passages relevant "
              "to a query, and a generator, a language model producing an answer conditioned on both the "
              "query and the retrieved passages. Because the answer is grounded in retrieved evidence "
              "rather than memorized weights, RAG reduces hallucination and lets the system's knowledge "
              "be updated by changing documents — no retraining required.")},
    {"id": "doc_7", "title": "Vector Databases and Similarity Search",
     "text": ("Once text is embedded, the vectors need storage supporting fast nearest-neighbor search at "
              "scale. Vector databases and libraries such as FAISS, Chroma, Pinecone, and Weaviate do "
              "exactly this: given a query vector they quickly return the most similar stored vectors "
              "using approximate or exact nearest-neighbor algorithms, the core of the retrieval step.")},
    {"id": "doc_8", "title": "Fine-tuning vs. Prompt Engineering vs. RAG",
     "text": ("Fine-tuning updates a model's weights with task-specific labeled data: powerful but "
              "expensive and redone whenever knowledge changes. Prompt engineering keeps the model "
              "frozen and crafts instructions: cheap but limited to what the model knows. RAG keeps the "
              "model frozen but retrieves external knowledge into the prompt at inference time, combining "
              "prompting's flexibility with access to knowledge the model never trained on.")},
    {"id": "doc_9", "title": "Hallucination in Large Language Models",
     "text": ("Hallucination is when a model generates fluent, confident text that is factually incorrect "
              "or unsupported. It happens because the model predicts plausible next tokens from learned "
              "patterns rather than looking anything up. RAG is one of the most effective mitigations, "
              "forcing answers to be based on retrieved, verifiable text rather than memory alone.")},
    {"id": "doc_10", "title": "Tokenization",
     "text": ("Before text reaches a Transformer it is split into tokens, usually subword units. "
              "Algorithms such as Byte-Pair Encoding, WordPiece, and SentencePiece learn a fixed "
              "vocabulary of common subword fragments, letting the model represent rare or unseen words "
              "as sequences of familiar pieces instead of needing an entry per word.")},
    {"id": "doc_11", "title": "Chunking Strategies for RAG",
     "text": ("Embedding and language models have limited context windows, and long documents are harder "
              "to retrieve precisely because one embedding must represent everything at once. So "
              "documents are split into smaller passages, or chunks, before embedding and indexing. "
              "Common strategies: fixed-size chunking by word or token count, sentence-based chunking, "
              "and sliding-window chunking with overlap so information near a boundary is not lost.")},
    {"id": "doc_12", "title": "Cosine Similarity",
     "text": ("Cosine similarity measures how similar two vectors are via the cosine of the angle between "
              "them, ignoring magnitude. It ranges from -1 to 1: 1 means the same direction (very similar "
              "meaning), 0 unrelated, -1 opposite. Because it only cares about direction, it is the "
              "standard metric for comparing text embeddings and equals a normalized dot product.")},
]

print(f"Knowledge base contains {len(KNOWLEDGE_BASE)} documents.")


Knowledge base contains 12 documents.


## 3. Chunking strategy

We split each document into overlapping word-window chunks. Overlap keeps information that
sits near a boundary from being cut in half between two chunks.


In [4]:
def chunk_text(text: str, chunk_size: int = 60, overlap: int = 15) -> list[str]:
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = end - overlap
    return chunks

# Build the flat list of chunks (each keeps a link back to its source doc)
chunks = []
for doc in KNOWLEDGE_BASE:
    for c in chunk_text(doc["text"]):
        chunks.append({"doc_id": doc["id"], "title": doc["title"], "text": c})

print(f"Created {len(chunks)} chunks from {len(KNOWLEDGE_BASE)} documents.")
print("Example chunk:\n", textwrap.fill(chunks[0]["text"], 90))


Created 17 chunks from 12 documents.
Example chunk:
 The Transformer is a neural network architecture introduced in 2017 that relies entirely
on attention mechanisms instead of recurrence or convolution to process sequences. Because
every token can directly attend to every other token, Transformers train in parallel
across the whole sequence, enabling much larger language models than earlier recurrent
architectures like LSTMs. Most modern LLMs, including BERT and GPT,


## 4. Generating embeddings

We use `all-MiniLM-L6-v2` (384-dim). We L2-normalize the vectors so that an inner-product
search is equivalent to cosine similarity.


In [5]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedder.encode(
    chunk_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)
print("Embeddings shape:", chunk_embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (17, 384)


## 5. Building a vector store with FAISS

`IndexFlatIP` does exact inner-product search. Because our vectors are normalized, inner
product = cosine similarity.


In [6]:
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)
print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}.")


FAISS index built with 17 vectors of dimension 384.


## 6. The retrieval function

Embed the query the same way, search the index, and return the top-k chunks with scores.


In [7]:
def retrieve(query: str, k: int = 3) -> list[dict]:
    query_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vec, k)
    return [{**chunks[i], "score": float(s)} for s, i in zip(scores[0], indices[0])]

test_query = "How does attention work inside a transformer?"
for r in retrieve(test_query, k=3):
    print(f"[{r['score']:.3f}] {r['title']}")
    print(textwrap.fill(r['text'], 90), "\n")


[0.545] The Attention Mechanism
Attention lets a model decide, for each token, how much to focus on every other token when
building its representation. It uses three learned projections: a Query, a Key, and a
Value. The similarity between a token's Query and every other token's Key produces
attention weights, used to compute a weighted sum over the Values. This captures long-
range dependencies, such 

[0.539] Transformer Architecture
The Transformer is a neural network architecture introduced in 2017 that relies entirely
on attention mechanisms instead of recurrence or convolution to process sequences. Because
every token can directly attend to every other token, Transformers train in parallel
across the whole sequence, enabling much larger language models than earlier recurrent
architectures like LSTMs. Most modern LLMs, including BERT and GPT, 

[0.366] Tokenization
Before text reaches a Transformer it is split into tokens, usually subword units.
Algorithms such as Byte-Pair Encodin

## 7. The generator model — a chat LLM, called directly

Here is the key difference from the original notebook. Instead of
`pipeline(...)[0]["generated_text"]`, we load a tokenizer and model and call
`model.generate()` ourselves, then decode. **No output dictionary key is involved,** so
this cannot raise `KeyError: 'generated_text'` regardless of your `transformers` version.

Model: **`Qwen/Qwen2.5-0.5B-Instruct`** — a small, modern, instruction-tuned chat model
that runs comfortably on Colab's free tier (and even on CPU, just slower). It uses a chat
template, which we apply with `tokenizer.apply_chat_template`.


In [9]:
GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading generator model (first run downloads weights)...")
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print("Generator ready.")


def generate(prompt: str, max_new_tokens: int = 256, system: str | None = None) -> str:
    """Run the chat LLM and return ONLY the newly generated text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Quick sanity check, no retrieval yet
print(generate("Translate to French: I love machine learning.", max_new_tokens=30))

Loading generator model (first run downloads weights)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generator ready.
Je aime la machine learning.


## 8. Prompt augmentation

This is where retrieval meets generation: we inject the retrieved chunks into the prompt
and instruct the model to answer **only** from that context.


In [10]:
def build_prompt(query: str, retrieved_chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"[{i + 1}] {c['text']}" for i, c in enumerate(retrieved_chunks)
    )
    return (
        "Answer the question using only the context below. "
        'If the answer is not in the context, say "I don\'t have enough information."\n\n'
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )

demo = retrieve("Why does RAG reduce hallucination?", k=3)
print(build_prompt("Why does RAG reduce hallucination?", demo))


Answer the question using only the context below. If the answer is not in the context, say "I don't have enough information."

Context:
[1] Hallucination is when a model generates fluent, confident text that is factually incorrect or unsupported. It happens because the model predicts plausible next tokens from learned patterns rather than looking anything up. RAG is one of the most effective mitigations, forcing answers to be based on retrieved, verifiable text rather than memory alone.

[2] weights, RAG reduces hallucination and lets the system's knowledge be updated by changing documents — no retraining required.

[3] RAG combines a retriever, which searches an external knowledge base for passages relevant to a query, and a generator, a language model producing an answer conditioned on both the query and the retrieved passages. Because the answer is grounded in retrieved evidence rather than memorized weights, RAG reduces hallucination and lets the system's knowledge be updated by ch

## 9. Putting it all together: the `RAGSystem` class

One reusable class that owns the embedder, the FAISS index, and the chat LLM, and exposes
`retrieve()` and `answer()`.


In [11]:
class RAGSystem:
    """A minimal Retrieval-Augmented Generation pipeline using a chat LLM directly."""

    SYSTEM = "You are a precise assistant that answers strictly from the provided context."

    def __init__(self, documents, embedder_model="all-MiniLM-L6-v2",
                 generator_model="Qwen/Qwen2.5-0.5B-Instruct",
                 chunk_size=60, overlap=15):
        self.documents = documents
        self.chunk_size = chunk_size
        self.overlap = overlap

        print("Loading embedding model...")
        self.embedder = SentenceTransformer(embedder_model, device=device)

        print("Loading generator model...")
        self.tokenizer = AutoTokenizer.from_pretrained(generator_model)
        self.model = AutoModelForCausalLM.from_pretrained(
            generator_model,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        ).to(device)
        self.model.eval()

        print("Chunking documents and building the vector index...")
        self._build_index()
        print(f"Ready. Indexed {len(self.chunks)} chunks from {len(documents)} documents.")

    def _chunk_text(self, text):
        words = text.split()
        out, start = [], 0
        while start < len(words):
            end = start + self.chunk_size
            out.append(" ".join(words[start:end]))
            if end >= len(words):
                break
            start = end - self.overlap
        return out

    def _build_index(self):
        self.chunks = []
        for doc in self.documents:
            for c in self._chunk_text(doc["text"]):
                self.chunks.append({"doc_id": doc["id"], "title": doc["title"], "text": c})
        texts = [c["text"] for c in self.chunks]
        emb = self.embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb)

    def _generate(self, prompt, max_new_tokens=256):
        messages = [{"role": "system", "content": self.SYSTEM},
                    {"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        return self.tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

    def retrieve(self, query, k=3):
        qv = self.embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        scores, idx = self.index.search(qv, k)
        return [{**self.chunks[i], "score": float(s)} for s, i in zip(scores[0], idx[0])]

    def build_prompt(self, query, retrieved):
        context = "\n\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(retrieved))
        return ("Answer the question using only the context below. "
                'If the answer is not in the context, say "I don\'t have enough information."\n\n'
                f"Context:\n{context}\n\nQuestion: {query}\nAnswer:")

    def answer(self, query, k=3, max_new_tokens=256):
        retrieved = self.retrieve(query, k=k)
        prompt = self.build_prompt(query, retrieved)
        generated = self._generate(prompt, max_new_tokens=max_new_tokens)
        return {"query": query, "answer": generated, "sources": retrieved}


In [12]:
# Build the full system from our knowledge base
rag = RAGSystem(KNOWLEDGE_BASE)


Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading generator model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Chunking documents and building the vector index...
Ready. Indexed 17 chunks from 12 documents.


In [13]:
def print_result(result: dict):
    print(f"Q: {result['query']}\n")
    print(f"A: {result['answer']}\n")
    print("Sources used:")
    for s in result["sources"]:
        print(f"  - {s['title']}  (score={s['score']:.3f})")
    print("\n" + "-" * 80 + "\n")

for question in [
    "What is the difference between BERT and GPT?",
    "Why does Retrieval-Augmented Generation reduce hallucination?",
    "What is cosine similarity used for?",
]:
    print_result(rag.answer(question))


Q: What is the difference between BERT and GPT?

A: Bert is an encoder-only Transformer trained with masked language modeling, while GPT is a decoder-only Transformer trained autoregressively.
I don't have enough information.

Sources used:
  - GPT: Generative Pretrained Transformer  (score=0.731)
  - BERT: Bidirectional Encoder Representations  (score=0.474)
  - Transformer Architecture  (score=0.470)

--------------------------------------------------------------------------------

Q: Why does Retrieval-Augmented Generation reduce hallucination?

A: RAG reduces hallucination by combining a retriever, which searches an external knowledge base for passages relevant to a query, and a generator, a language model producing an answer conditioned on both the query and the retrieved passages.
This can be seen in the context where it states that RAG "combines a retriever, which searches an external knowledge base for passages relevant to a query, and a generator, a language model producing an

## 10. RAG vs. no-RAG: does retrieval actually help?

Same question asked two ways: the bare model from memory, versus the model given retrieved
context. Watch how grounding changes the answer.


In [14]:
comparison_query = "What chunking strategy helps avoid losing information at chunk boundaries?"

# 1) No-RAG: ask the bare model directly, no retrieved context
no_rag_output = generate(comparison_query, max_new_tokens=120)

# 2) RAG: retrieve + augment + generate
rag_result = rag.answer(comparison_query)

print("=== WITHOUT RAG (model answers from memory alone) ===")
print(textwrap.fill(no_rag_output, 90))
print()
print("=== WITH RAG (model answers using retrieved context) ===")
print(textwrap.fill(rag_result["answer"], 90))
print()
print("Context that was retrieved and given to the model:")
for s in rag_result["sources"]:
    print(f"  - {s['title']}")


=== WITHOUT RAG (model answers from memory alone) ===
Chunking is a technique used in data processing to break down large datasets into smaller,
more manageable pieces or chunks. This approach can help prevent information loss and
improve the efficiency of data analysis and retrieval. Here are some strategies that can
be effective for chunking:  1. **Data Partitioning**: Divide your dataset into smaller
subsets (chunks) based on certain criteria such as size, type, or relevance. For example,
you might divide a large dataset into smaller files or folders.  2. **Hierarchical
Chunking**: Organize your data into nested structures like directories or subdirectories
within larger directories.

=== WITH RAG (model answers using retrieved context) ===
Sentence-based chunking

Context that was retrieved and given to the model:
  - Chunking Strategies for RAG
  - Chunking Strategies for RAG
  - Tokenization


## 11. Evaluating retrieval quality

Before judging the generator, check the retriever: for each test query, is the correct
source document among the top-k retrieved chunks? This is Precision@k.


In [15]:
test_cases = [
    {"query": "How do transformers use attention to relate words to each other?", "expected_title": "The Attention Mechanism"},
    {"query": "Why do language models sometimes make up facts that sound plausible?", "expected_title": "Hallucination in Large Language Models"},
    {"query": "How should long documents be split before they are embedded?", "expected_title": "Chunking Strategies for RAG"},
    {"query": "Which model architecture is best suited for open-ended text generation?", "expected_title": "GPT: Generative Pretrained Transformer"},
    {"query": "What metric is commonly used to compare two embedding vectors?", "expected_title": "Cosine Similarity"},
    {"query": "What is the difference between fine-tuning and prompting a model?", "expected_title": "Fine-tuning vs. Prompt Engineering vs. RAG"},
]

def evaluate_retrieval(cases, k=3):
    hits = 0
    for case in cases:
        titles = [r["title"] for r in rag.retrieve(case["query"], k=k)]
        is_hit = case["expected_title"] in titles
        hits += int(is_hit)
        print(f"{'✅' if is_hit else '❌'}  expected: {case['expected_title']:<45} | got: {titles}")
    return hits / len(cases)

acc = evaluate_retrieval(test_cases, k=3)
print(f"\nPrecision@3 across {len(test_cases)} test queries: {acc * 100:.1f}%")


✅  expected: The Attention Mechanism                       | got: ['Tokenization', 'Transformer Architecture', 'The Attention Mechanism']
✅  expected: Hallucination in Large Language Models        | got: ['Hallucination in Large Language Models', 'Tokenization', 'Transformer Architecture']
✅  expected: Chunking Strategies for RAG                   | got: ['Chunking Strategies for RAG', 'Chunking Strategies for RAG', 'Tokenization']
❌  expected: GPT: Generative Pretrained Transformer        | got: ['Transformer Architecture', 'Transformer Architecture', 'Retrieval-Augmented Generation (RAG)']
✅  expected: Cosine Similarity                             | got: ['Cosine Similarity', 'Word and Sentence Embeddings', 'Vector Databases and Similarity Search']
✅  expected: Fine-tuning vs. Prompt Engineering vs. RAG    | got: ['Fine-tuning vs. Prompt Engineering vs. RAG', 'Fine-tuning vs. Prompt Engineering vs. RAG', 'The Attention Mechanism']

Precision@3 across 6 test queries: 83.3%


## 14. Exercises for students

1. Add 3 new documents to `KNOWLEDGE_BASE` and re-run — confirm they get retrieved.
2. Change `chunk_size` and `overlap` and observe the effect on Precision@3.
3. Ask a question whose answer is **not** in the knowledge base. Does the model correctly
   say it lacks information?

